# QUBO feature selection (Romero et al. 2025)

Reimplementation of Eq. (4) from Romero, Gupta, Gatlin, Chapkin, Cai,
*Quantum Machine Intelligence* (2025) 7:114, applied to GEO **GSE308682**.

$$
\min_{F} \left[ -\alpha \sum_{i} I_{i} F_{i} + (1-\alpha)\sum_{i,j} R_{ij} F_{i} F_{j} \right], \quad Q = (1-\alpha)R - \alpha\,\mathrm{diag}(I)
$$

Logic lives in `qubo_model.py`, `data_loader.py`, `qubo_experiment.py`. This notebook only configures and runs the experiment.

Set **`QUBO_DATA_DIR`** in the first code cell (or `export QUBO_DATA_DIR=...`) to the folder that contains:

- `GSE308682_filtered_matrix.mtx.gz`
- `GSE308682_filtered_features.tsv.gz`
- `GSE308682_filtered_barcodes.tsv.gz`
- `GSE308682_feature_reference.csv.gz`

These 10x files are not in git. Optional: **`QUBO_REPO_DIR`** if the kernel cwd is not this repo.

Comparison vs the paper: [`docs/compare-romero-2025.html`](docs/compare-romero-2025.html) · [中文](docs/compare-romero-2025.zh.html).


In [ ]:
# Core deps. Optional QBoson: pip install kaiwu==1.3.1 torch
# pip install git+https://github.com/qboson/kaiwu-pytorch-plugin.git
%pip install -q numpy scikit-learn matplotlib seaborn dwave-ocean-sdk scanpy ipywidgets tqdm


In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

# --- paths: set these before loading real data ---
# QUBO_REPO_DIR: folder that contains qubo_model.py (default: notebook cwd).
os.environ.setdefault("QUBO_REPO_DIR", str(Path.cwd()))
# QUBO_DATA_DIR: folder with GEO GSE308682 10x files:
#   GSE308682_filtered_matrix.mtx.gz
#   GSE308682_filtered_features.tsv.gz
#   GSE308682_filtered_barcodes.tsv.gz
#   GSE308682_feature_reference.csv.gz
# Uncomment and edit, or export QUBO_DATA_DIR in the shell:
# os.environ["QUBO_DATA_DIR"] = "/path/to/gse308682_dir"

ROOT = Path(os.environ["QUBO_REPO_DIR"]).expanduser().resolve()
if not (ROOT / "qubo_model.py").exists():
    raise FileNotFoundError(
        f"qubo_model.py not found in QUBO_REPO_DIR={ROOT}. "
        "Set os.environ['QUBO_REPO_DIR'] to this repository, or run the notebook from the repo root."
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
from qubo_model import (
    HAS_DWAVE,
    HAS_KAIWU,
    HAS_KAIWU_PLUGIN,
    HAS_TABU,
    available_solvers,
    compute_mutual_information_matrix,
    solve_qubo_target_k,
    solver_banner,
)
from qubo_experiment import (
    compare_with_lasso_rfr,
    load_experiment,
    print_regression_mse,
    target_cardinality,
)

print(f"Importing Python modules from {ROOT}")
print(f"QUBO_DATA_DIR={os.environ.get('QUBO_DATA_DIR', '(unset)')}")
print(f"Ocean tabu={HAS_TABU}, Ocean SA={HAS_DWAVE}, Kaiwu SDK={HAS_KAIWU}, torch plugin={HAS_KAIWU_PLUGIN}")


## Solvers

`dwave-ocean-sdk` is already used. **`tabu` / `sa` are classical CPU samplers**, not a QPU.
`leap` is D-Wave’s cloud hybrid solver. `kaiwu_cim` is QBoson’s photonic CIM.

| `SOLVER` | Stack | Hardware | Env |
| --- | --- | --- | --- |
| `tabu` | Ocean | classical CPU | — |
| `sa` | Ocean | classical CPU | — |
| `leap` | Ocean | Leap hybrid | `DWAVE_API_TOKEN` |
| `custom_sa` | this repo | classical CPU | — |
| `kaiwu_sa` / `kaiwu_tabu` | kaiwu SDK | classical CPU | optional license |
| `kaiwu_cim` | kaiwu CIM | photonic QPU | `KAIWU_USER_ID`, `KAIWU_SDK_CODE` |


In [ ]:
# Romero et al. §4.1-style settings. Implementations: data_loader.py, qubo_model.py
# T: "pseudotime" = scanpy DPT. "gene" = held-out RUNX1 Pearson residual.
# N_TOP_GENES: paper used ~5,000 HVGs. Pairwise MI is O(p²); tqdm shows progress.
# Tabu: Ocean default timeout is 20 ms/read (too small at p=5000). This repo
# overrides it; set QUBO_TABU_TIMEOUT_MS to change (milliseconds).
USE_REAL_DATA = True
N_TOP_GENES = 5000
TARGET_MODE = "pseudotime"
ROOT_GENE = "HBE1"          # DPT iroot = cell with min residual of this gene
SOLVER = "tabu"             # tabu | sa | leap | custom_sa | kaiwu_sa | kaiwu_tabu | kaiwu_cim
K_TARGET = 50               # paper real-data cardinality

import qubo_model as _qubo_model
print(solver_banner(SOLVER))
print("qubo_model loaded from", _qubo_model.__file__)
print("Available solvers:")
for name, meta in available_solvers().items():
    mark = "yes" if meta["installed"] else "no"
    print(f"  {name:12} installed={mark:3}  {meta['kind']:16}  {meta['stack']}")

X, y, true_features, feature_names = load_experiment(
    use_real_data=USE_REAL_DATA,
    n_top_genes=N_TOP_GENES,
    target_mode=TARGET_MODE,
    root_gene=ROOT_GENE,
)


In [ ]:
# Discrete MI: qubo_model.compute_mutual_information_matrix (tqdm on bins, I, R)
I, R = compute_mutual_information_matrix(X, y, n_bins=10)
print(f"Importance I (first 5): {I[:5]}")
print(f"Redundancy R (5x5):\n{R[:5, :5]}")


In [ ]:
K = target_cardinality(X.shape[1], k=K_TARGET)
print(f"Target cardinality K={K}")


In [ ]:
selected_features_qubo, energy, alpha, Q, report = solve_qubo_target_k(
    I, R, k=K, solver=SOLVER
)
selected_idx_qubo = np.where(selected_features_qubo == 1)[0]
print(f"α*={alpha:.4f}, energy={energy:.4f}, |F*|={len(selected_idx_qubo)}")
print(f"QUBO selected indices: {selected_idx_qubo}")
if feature_names is not None:
    print("QUBO selected genes:", [feature_names[i] for i in selected_idx_qubo])

# LASSO/RF stay at paper k. Never retarget them to a failed |F*|.
k_compare = K
if not report["accepted"]:
    print(
        f"WARNING: QUBO failed energy/cardinality acceptance "
        f"(energy={energy:.4f}, |F*|={len(selected_idx_qubo)}, k={K}). "
        "LASSO/RF below use target K, not |F*|. "
        "Do not treat MSE as proof that Eq. (4) was minimized."
    )
elif len(selected_idx_qubo) != K:
    print(
        f"WARNING: |F*|={len(selected_idx_qubo)} != target K={K} "
        "(within tolerance but not exact)."
    )


In [ ]:
# LASSO path (~K nonzeros) and random forest. Overlaps printed here.
selected_idx_lasso, selected_idx_rf = compare_with_lasso_rfr(
    X, y, I, true_features, selected_idx_qubo, K=k_compare, feature_names=feature_names
)


In [ ]:
print_regression_mse(X, y, selected_idx_qubo, selected_idx_lasso, selected_idx_rf)


## After a run

Open [`docs/compare-romero-2025.html`](docs/compare-romero-2025.html) or [中文](docs/compare-romero-2025.zh.html) for the last analyzed comparison vs the paper.

A healthy unconstrained Eq. (4) solve has **negative energy** and **|F\*| ≈ K** (50). Large positive energy with |F\*| ≈ p/2 means tabu/α-search failed to find a sparse minimizer — switch solver (`sa`, `leap`, `kaiwu_cim`) or inspect the α bisection tqdm postfix (`n_sel`).
